In [6]:
import os
import vertexai

# Set your Qwiklabs Google Cloud Project details
PROJECT_ID = "qwiklabs-gcp-02-75ff770f7744"  # Update if your temporary project ID changed
LOCATION = "us-central1"
MODEL_LOCATION = "global"
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging"

# Set global project environment variable
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)
print("Vertex AI Initialized Successfully.")

Vertex AI Initialized Successfully.


In [7]:
from google.adk.agents import Agent
from google.adk.tools import google_search

# 1. Define the Agent using the Google Search tool
search_agent = Agent(
    name="GoogleSearchAgent",
    model="gemini-2.0-flash",
    description="Provides Google Search Results.",
    instruction="You are a helpful assistant. Use the Google Search tool to find up-to-date information and answer user queries accurately.",
    tools=[google_search]
)

print("Agent defined successfully.")

Agent defined successfully.


In [9]:
from vertexai import agent_engines

# 2. Wrap the agent into an executable app format for the cloud
app = agent_engines.AdkApp(agent=search_agent)

print("Agent wrapped as AdkApp successfully.")

Agent wrapped as AdkApp successfully.


In [10]:
from vertexai import types
from vertexai import agent_engines

# 3. Create the Vertex AI Client
client = vertexai.Client(
    project=PROJECT_ID,
    location=LOCATION,
)

print("Deploying Google Search Agent... (This takes 3-5 minutes)")

# 4. Deploy to Vertex AI Agent Engines
remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]",
            "google-adk",
            "google-genai"
        ],
        "staging_bucket": STAGING_BUCKET,
        "env_vars": {"GOOGLE_CLOUD_LOCATION": "global"},
        "identity_type": types.IdentityType.AGENT_IDENTITY,
        "display_name": "Google Search Agent",
        "description": "This agent can search Google and return answers. ",
    }
)

print("\n✅ Agent Successfully Deployed!")
print(f"Full Resource Name: {remote_agent.api_resource.name}")

INFO:vertexai_genai.agentengines:Identified the following requirements: {'pydantic': '2.13.4', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '2.2.0'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk', 'google-genai', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
INFO:vertexai_genai.agentengines:Using bucket qwiklabs-gcp-02-75ff770f7744-agent-staging


Deploying Google Search Agent... (This takes 3-5 minutes)


INFO:vertexai_genai.agentengines:Wrote to gs://qwiklabs-gcp-02-75ff770f7744-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://qwiklabs-gcp-02-75ff770f7744-agent-staging/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://qwiklabs-gcp-02-75ff770f7744-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-02-75ff770f7744&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%224187811366658113536%22.
INFO:vertexai_genai.agentengines:Agent Engine created. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines.get(name='projects/810068441016/locations/us-central1/rea


✅ Agent Successfully Deployed!
Full Resource Name: projects/810068441016/locations/us-central1/reasoningEngines/4187811366658113536


In [11]:
import os
import vertexai
from vertexai import types, agent_engines
from google.adk.agents import Agent
from google.adk.tools import google_search

# 1. Environment Setup
PROJECT_ID = "qwiklabs-gcp-02-75ff770f7744"
LOCATION = "us-central1"
MODEL_LOCATION = "global"  # The critical fix for model routing!
STAGING_BUCKET = f"gs://{PROJECT_ID}-agent-staging"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket=STAGING_BUCKET)
client = vertexai.Client(project=PROJECT_ID, location=LOCATION)

# 2. Agent Definition
search_agent = Agent(
    name="GoogleSearchAgent",
    model="gemini-2.0-flash",
    description="Provides Google Search Results.",
    instruction="You are a helpful assistant. Use the Google Search tool to find up-to-date information.",
    tools=[google_search]
)

# 3. Wrap as ADK App
app = agent_engines.AdkApp(agent=search_agent)

print("Deploying Google Search Agent with global model routing... (Takes 3-5 minutes)")

# 4. Deploy using Agent Engines
remote_agent = client.agent_engines.create(
    agent=app,
    config={
        "requirements": [
            "google-cloud-aiplatform[agent_engines,adk]",
            "google-adk",
            "google-genai"
        ],
        "staging_bucket": STAGING_BUCKET,
        "env_vars": {"GOOGLE_CLOUD_LOCATION": MODEL_LOCATION}, # Forces global LLM lookup
        "identity_type": types.IdentityType.AGENT_IDENTITY,
        "display_name": "Google Search Agent",
        "description": "This agent can search Google and return answers."
    }
)

# --- CAPTURE SCREENSHOT 1 HERE ---
print(f"\n✅ Agent Successfully Deployed!")
print(f"Full Resource Name: {remote_agent.api_resource.name}")
print("\n" + "="*50 + "\n")

# 5. Live Query Test
print(f"Querying live Google Search agent...\n")
async for event in remote_agent.async_stream_query(
    user_id="prod-user-01",
    message="Who won the most recent World Series and how many games did the series go?"
):
    print(event)
# --- CAPTURE SCREENSHOT 2 HERE ---

INFO:vertexai_genai.agentengines:Identified the following requirements: {'pydantic': '2.13.4', 'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '2.2.0'}
INFO:vertexai_genai.agentengines:The following requirements are appended: {'pydantic==2.13.4', 'cloudpickle==3.1.2'}
INFO:vertexai_genai.agentengines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'google-adk', 'google-genai', 'pydantic==2.13.4', 'cloudpickle==3.1.2']
INFO:vertexai_genai.agentengines:Using bucket qwiklabs-gcp-02-75ff770f7744-agent-staging


Deploying Google Search Agent with global model routing... (Takes 3-5 minutes)


INFO:vertexai_genai.agentengines:Wrote to gs://qwiklabs-gcp-02-75ff770f7744-agent-staging/agent_engine/agent_engine.pkl
INFO:vertexai_genai.agentengines:Writing to gs://qwiklabs-gcp-02-75ff770f7744-agent-staging/agent_engine/requirements.txt
INFO:vertexai_genai.agentengines:Creating in-memory tarfile of extra_packages
INFO:vertexai_genai.agentengines:Writing to gs://qwiklabs-gcp-02-75ff770f7744-agent-staging/agent_engine/dependencies.tar.gz
INFO:vertexai_genai.agentengines:Using agent framework: google-adk
INFO:vertexai_genai.agentengines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-02-75ff770f7744&query=resource.type%3D%22aiplatform.googleapis.com%2FReasoningEngine%22%0Aresource.labels.reasoning_engine_id%3D%226894474742707781632%22.
INFO:vertexai_genai.agentengines:Agent Engine created. To use it in another session:
INFO:vertexai_genai.agentengines:agent_engine=client.agent_engines.get(name='projects/810068441016/locations/us-central1/rea


✅ Agent Successfully Deployed!
Full Resource Name: projects/810068441016/locations/us-central1/reasoningEngines/6894474742707781632


Querying live Google Search agent...

{'error_code': 'NOT_FOUND', 'error_message': "404 NOT_FOUND. {'error': {'code': 404, 'message': 'Publisher model `projects/qwiklabs-gcp-02-75ff770f7744/locations/global/publishers/google/models/gemini-2.0-flash` was not found or your project does not have access to it. Ensure you are using a valid model name and that the model is available in the specified region. For more information, see: https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations.', 'status': 'NOT_FOUND'}}", 'invocation_id': 'e-7c91e50c-39f6-4660-96d6-937fa29c48e1', 'author': 'GoogleSearchAgent', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}, 'requested_tool_confirmations': {}}, 'node_info': {'path': 'GoogleSearchAgent@1'}, 'id': 'abdd6aac-942f-4699-bc18-4f0cfd472429', 'timestamp': 179200